In [1]:
import os
import numpy as np
import pandas as pd
from langdetect import detect
import emoji
import re
import ollama
import json

np.set_printoptions(suppress=True)

# Data loading

In [2]:
data_dir = "data"
reviews_file = os.path.join(data_dir, "reviews.csv")
reviews_data = pd.read_csv(reviews_file)

# Preprocesing

In [3]:
def clean_text(text):
    text = re.sub(r'⭐{5}', ' five out of five, great', text)
    text = re.sub(r'⭐{4}', ' four out of five, almost great ', text)
    text = re.sub(r'⭐{3}', ' three out of five, not bad ', text)
    text = re.sub(r'⭐{2}', ' two out of five, could be better ', text)
    text = re.sub(r'⭐{1}', ' one out of five, improvement needed ', text)

    emoji_map = {
        "👎": " would not recommend ",
        "😡": " i'm angry",
        "🔥": " great",
        "✅": " asset",
        "❌": " flaw"
    }

    for emoji_char, words in emoji_map.items():
        text = text.replace(emoji_char, words)

    text = emoji.demojize(text, delimiters=(" ", " "))
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.replace("<br/>", " ")

    return text

def is_valid_english_review(text):
    if isinstance(text, str) and len(text.split()) > 50:
        try:
            return detect(text) == "en"
        except:
            return False
    return False

# Create Gemini Client

In [7]:
LOCAL_MODEL = "llama3.1"

def generate_summary_local(payload_list):
    formatted_input = "\n".join([f"ID {i}: {txt}" for i, txt in payload_list])
    prompt = f"""
    Summarize each customer review in one sentence.
    Return the results ONLY as a JSON array of objects.
    Each object must have "id" (integer) and "summary" (string).

    REVIEWS:
    {formatted_input}
    """

    try:
        response = ollama.generate(
            model="llama3.1",
            prompt=prompt,
            format="json",
            options={"temperature": 0.1, "num_ctx": 2048},
        )
        return json.loads(response['response'])
    except Exception as e:
        print(f"Error: {e}")
        return None

# Summarization pipeline

In [6]:
reviews_to_be_processed = 50000
df_subset = reviews_data.iloc[:reviews_to_be_processed].copy()
df_all_english = df_subset[reviews_data["comments"].apply(is_valid_english_review).copy()]
df_all_english.shape

/tmp/ipykernel_134160/1280299674.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_all_english = df_subset[reviews_data["comments"].apply(is_valid_english_review).copy()]


(12550, 7)

In [12]:
n = 1000
batch_size = 5

for k in range(12):
    start = k * n
    end = (k + 1) * n
    df_chunk = df_all_english.iloc[start:end].copy()

    for i in range(0, len(df_chunk), batch_size):
        print(f"Batch {i // batch_size + 1}/{n // batch_size}")
        batch = df_chunk.iloc[i : i + batch_size]
        payload = list(zip(batch.index, batch["comments"].apply(clean_text)))

        results = generate_summary_local(payload)
        if results:
            try:
                if isinstance(results["reviews"], list):
                    for item in results["reviews"]:
                        try:
                            res_id = int(item.get("id"))
                            res_sum = item.get("summary")
                            df_chunk.at[res_id, "summary"] = res_sum
                        except Exception as e:
                            print(e)
            except Exception as e:
                print(e)

    df_chunk.to_csv(f"summaries3/summaries{k}.csv", index=False)

Batch 1/200
Batch 2/200
Batch 3/200
Batch 4/200
Batch 5/200
Batch 6/200
Batch 7/200
Batch 8/200
Batch 9/200
Batch 10/200
Batch 11/200
Batch 12/200
Batch 13/200
Batch 14/200
Batch 15/200
Batch 16/200
Batch 17/200
Batch 18/200
Batch 19/200
Batch 20/200
Batch 21/200
Batch 22/200
Batch 23/200
Batch 24/200
Batch 25/200
Batch 26/200
Batch 27/200
Batch 28/200
Batch 29/200
Batch 30/200
Batch 31/200
Batch 32/200
Batch 33/200
Batch 34/200
Batch 35/200
Batch 36/200
Batch 37/200
Batch 38/200
Batch 39/200
Batch 40/200
Batch 41/200
Batch 42/200
Batch 43/200
Batch 44/200
Batch 45/200
Batch 46/200
Batch 47/200
Batch 48/200
Batch 49/200
Batch 50/200
Batch 51/200
Batch 52/200
Batch 53/200
Batch 54/200
Batch 55/200
Batch 56/200
Batch 57/200
Batch 58/200
Batch 59/200
Batch 60/200
Batch 61/200
Batch 62/200
Batch 63/200
Batch 64/200
Batch 65/200
Batch 66/200
Batch 67/200
Batch 68/200
Batch 69/200
Batch 70/200
Batch 71/200
Batch 72/200
Batch 73/200
Batch 74/200
Batch 75/200
Batch 76/200
Batch 77/200
Batch 78